In [1]:
## Load required libraries
!pip install -q transformers datasets sentencepiece sacremoses evaluate accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 13.1 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 39.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 109.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 88.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 34.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 14.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback
)
import evaluate


In [8]:
## Check GPU availability
## Set device configuration

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)


cuda


In [9]:
## Load dataset
dataset = load_dataset("cfilt/iitb-english-hindi")


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/190M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/85.7k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/500k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1659083 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/520 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2507 [00:00<?, ? examples/s]

In [13]:
## Load dataset
dataset["train"] = dataset["train"].shuffle(seed=42).select(range(50000))
dataset["validation"] = dataset["validation"].shuffle(seed=42).select(range(520))
dataset["test"] = dataset["test"].shuffle(seed=42).select(range(2500))


In [14]:
## login to hugging face
!pip install -q huggingface_hub


In [18]:
from huggingface_hub import login
login()


In [19]:
model_name = "ai4bharat/indictrans2-en-indic-dist-200M"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    trust_remote_code=True
).to(device)


tokenizer_config.json:   0%|          | 0.00/1.11k [00:00<?, ?B/s]

tokenization_indictrans.py:   0%|          | 0.00/8.04k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-en-indic-dist-200M:
- tokenization_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


dict.SRC.json:   0%|          | 0.00/645k [00:00<?, ?B/s]

dict.TGT.json:   0%|          | 0.00/3.39M [00:00<?, ?B/s]

...

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-en-indic-dist-200M:
- configuration_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_indictrans.py:   0%|          | 0.00/79.8k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-en-indic-dist-200M:
- modeling_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

In [24]:
# Disable cache (prevents crash)
model.config.use_cache = False

# Save GPU memory
model.gradient_checkpointing_enable()


You are using an old version of the checkpointing format that is deprecated (We will also silently ignore `gradient_checkpointing_kwargs` in case you passed it).Please update to the new format on your modeling file. To use the new format, you need to completely remove the definition of the method `_set_gradient_checkpointing` in your model.


In [26]:
## Preprocess dataset for translation
SRC_LANG = "eng_Latn"
TGT_LANG = "hin_Deva"

max_input_length = 128
max_target_length = 128

def preprocess_function(examples):
    inputs = [
        f"{SRC_LANG} {TGT_LANG} {item['en']}"
        for item in examples["translation"]
    ]
    targets = [item["hi"] for item in examples["translation"]]

    model_inputs = tokenizer(
        inputs,
        truncation=True,
        padding="max_length",
        max_length=max_input_length
    )

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            targets,
            truncation=True,
            padding="max_length",
            max_length=max_target_length
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


In [27]:
## Load pre-trained IndicTrans2 model and tokenizer
tokenized_datasets = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)


Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:3951: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/520 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

In [31]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)


In [33]:
## Define training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/indictrans2_finetune",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=1e-5,
    weight_decay=0.01,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=16,   # effective batch = 64

    num_train_epochs=4,               # 🔴 finishes in time
    fp16=True,

    predict_with_generate=False,      # avoid crash
    prediction_loss_only=True,        # avoid OOM

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    save_total_limit=2,
    report_to="none"
)


In [34]:
# Define early stopping
early_stopping = EarlyStoppingCallback(
    early_stopping_patience=1,
    early_stopping_threshold=0.01
)


In [35]:
## Start model fine-tuning
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    callbacks=[early_stopping]
)


/tmp/ipykernel_47/3055854984.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [36]:
##empty cache necessary for indictrans2
torch.cuda.empty_cache()
trainer.train()


/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,No log,5.426890
2,7.468600,2.919970
3,4.050700,1.597356
4,2.107200,1.241831


/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/

TrainOutput(global_step=1876, training_loss=3.9170460975500565, metrics={'train_runtime': 6846.9655, 'train_samples_per_second': 17.526, 'train_steps_per_second': 0.274, 'total_flos': 1.220378886144e+16, 'train_loss': 3.9170460975500565, 'epoch': 4.0})

In [37]:
#evaluate loss
test_results = trainer.evaluate(tokenized_datasets["test"])

print(test_results)


{'eval_loss': 1.2906715869903564, 'eval_runtime': 67.1046, 'eval_samples_per_second': 37.255, 'eval_steps_per_second': 18.628, 'epoch': 4.0}


In [41]:
load_best_model_at_end=True
metric_for_best_model="eval_loss"


In [42]:
##push mode to hugging face for  testing
final_model_dir = "/kaggle/working/final_model"

trainer.save_model(final_model_dir)
tokenizer.save_pretrained(final_model_dir)


('/kaggle/working/final_model/tokenizer_config.json',
 '/kaggle/working/final_model/special_tokens_map.json',
 '/kaggle/working/final_model/dict.SRC.json',
 '/kaggle/working/final_model/dict.TGT.json',
 '/kaggle/working/final_model/model.SRC',
 '/kaggle/working/final_model/model.TGT',
 '/kaggle/working/final_model/added_tokens.json')

In [44]:
load_best_model_at_end=True
metric_for_best_model="eval_loss"


TypeError: Trainer.create_model_card() got an unexpected keyword argument 'repo_id'

In [48]:
training_args.hub_model_id = REPO_ID


In [49]:
trainer.push_to_hub(
    commit_message="Fine-tuned IndicTrans2 on IITB En-Hi dataset (Kaggle, 20k samples, 4 epochs)"
)


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/deepanshumiglani0408/indictrans2_finetune/commit/07e70fb9d79e2cf959ccd8b063476f5daf770b0e', commit_message='Fine-tuned IndicTrans2 on IITB En-Hi dataset (Kaggle, 20k samples, 4 epochs)', commit_description='', oid='07e70fb9d79e2cf959ccd8b063476f5daf770b0e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/deepanshumiglani0408/indictrans2_finetune', endpoint='https://huggingface.co', repo_type='model', repo_id='deepanshumiglani0408/indictrans2_finetune'), pr_revision=None, pr_num=None)

NotImplementedError: Cannot copy out of meta tensor; no data! Please use torch.nn.Module.to_empty() instead of torch.nn.Module.to() when moving module from meta to a different device.

ImportError: To be able to use evaluate-metric/sacrebleu, you need to install the following dependencies['sacrebleu'] using 'pip install sacrebleu' for instance'